# Rimworld-Statistics：実セーブ解析

**Google Colab ブラウザ版・CPUランタイム専用**。既存の .NET 10 `RealSaveImport` と `SaveImportBenchmark --dir` で `.rws` / `.rws.old` を読み取り、整合性・取り込み時間・割当量を匿名集計します。

1. 「ランタイム → ランタイムのタイプを変更」で **CPU** を選択。
2. 左の「ファイル」へ `.rws` をアップロードするか、Google Drive上の保存フォルダを指定する。
3. **ランタイム → すべてのセルを実行**。結果は `/content/rws-colab-output/` に保存。

**機密保護**：生のセーブをGitHub・ノートブック結果・公開レポートには含めません。Driveを使う場合もローカルVMにコピーしてから読み取り、元ファイルは変更しません。Colab VMと認可したGoogle Driveには、セーブ内容が存在する点に注意してください。結果を共有する前に内容を確認してください。

## 1. 入力を指定

直接アップロードした場合は `SAVE_SOURCE='/content'`。Driveの場合は `USE_DRIVE=True` にして、例えば `SAVE_SOURCE='/content/drive/MyDrive/RimWorld/Saves'` を指定します。対象は指定フォルダ**直下**のファイルのみです。

In [ ]:
SAVE_SOURCE = "/content"  # @param {type:"string"}
USE_DRIVE = False  # @param {type:"boolean"}
REPEATS = 3  # @param {type:"slider",min:1,max:10,step:1}
REPEATS = int(REPEATS)
assert 1 <= REPEATS <= 10

## 2. GitHubと.NET 10を準備

公開リポジトリの最新mainだけを取得します。初回は.NET SDKのインストールが必要です。SDK、GitHubからの取得はインターネット接続を必要とします。

In [ ]:
import csv, hashlib, io, json, os, shutil, statistics, subprocess, urllib.request
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

REPO = Path("/content/rws-colab-repository")
STAGE = Path("/content/rws-colab-private/saves")
OUT = Path("/content/rws-colab-output")
OUT.mkdir(parents=True, exist_ok=True)

def command(args, cwd=None, private=False, timeout=900):
    completed = subprocess.run([str(a) for a in args], cwd=cwd, env=os.environ.copy(),
                               capture_output=True, text=True, errors="replace", timeout=timeout)
    if completed.returncode != 0:
        if private:
            raise RuntimeError("非公開セーブの処理に失敗しました。機密保護のため詳細な標準エラーは表示しません。")
        raise RuntimeError(f"コマンド失敗 (終了コード {completed.returncode}): {completed.stderr[-1400:]}")
    return completed.stdout.strip()

def setup():
    if not (REPO / ".git").is_dir():
        command(["git", "clone", "--depth", "1",
                 "https://github.com/pengin0503/Rimworld-Statistics.git", REPO], timeout=240)
    else:
        command(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], timeout=240)
        command(["git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD"])
    dotnet_root = Path("/content/rws-dotnet")
    binary = dotnet_root / "dotnet"
    if not binary.is_file():
        installer = Path("/content/rws-dotnet-install.sh")
        urllib.request.urlretrieve("https://dot.net/v1/dotnet-install.sh", installer)
        command(["bash", installer, "--channel", "10.0", "--install-dir", dotnet_root], timeout=900)
    os.environ["DOTNET_ROOT"] = str(dotnet_root)
    os.environ["PATH"] = str(dotnet_root) + os.pathsep + os.environ["PATH"]
    os.environ["DOTNET_CLI_TELEMETRY_OPTOUT"] = "1"
    os.environ["DOTNET_SKIP_FIRST_TIME_EXPERIENCE"] = "1"
    version = command(["dotnet", "--version"])
    assert version.startswith("10."), f".NET 10 が必要です: {version}"
    sha = command(["git", "-C", REPO, "rev-parse", "HEAD"])
    print("検証対象:", sha[:12], "/ .NET", version)
    return sha

COMMIT_SHA = setup()

## 3. セーブをVM内の専用フォルダへ複製

Driveの多数の小ファイルを直接走査せず、コピー後に解析します。コピー元とコピー後をSHA-256で照合し、元ファイルの更新はしません。元のファイル名は画面に表示しません。空き容量を確保してください。

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

source_dir = Path(SAVE_SOURCE).expanduser()
if not source_dir.is_dir():
    raise FileNotFoundError("入力フォルダがありません。SAVE_SOURCE を確認してください。")
inputs = sorted(p for p in source_dir.iterdir()
                if p.is_file() and not p.is_symlink()
                and p.name.lower().endswith((".rws", ".rws.old")))
if not inputs:
    raise ValueError("入力フォルダ直下に .rws / .rws.old がありません。")
if len(inputs) > 1000:
    raise ValueError("安全のため1000ファイル以下に分けてください。")
bytes_total = sum(p.stat().st_size for p in inputs)
if shutil.disk_usage("/content").free < bytes_total * 1.2:
    raise RuntimeError("VMの空き容量が不足しています。元ファイルの合計容量を確認してください。")

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_hashes = [(p, sha256(p)) for p in inputs]
source_real = Path(os.path.realpath(source_dir))
stage_real = Path(os.path.realpath(STAGE))
if source_real == stage_real or source_real.is_relative_to(stage_real):
    raise ValueError("SAVE_SOURCE に作業用ステージフォルダ自身またはその配下は指定できません。")
if STAGE.exists():
    shutil.rmtree(STAGE)  # Colab内の専用一時コピーだけを消す
STAGE.mkdir(parents=True)
for path, digest in source_hashes:
    destination = STAGE / path.name
    shutil.copy2(path, destination)
    if sha256(destination) != digest:
        raise RuntimeError("コピー元とコピー先のハッシュが一致しません。")
print(f"準備完了: {len(inputs)}ファイル / {bytes_total / (1024**2):,.1f} MiB")

## 4. 実セーブの読み取りを検証

既存のC# `SaveReader`による読み取り結果です。`parsed_ok` と `unidentifiable` は別に計数し、識別できなかったセーブを成功扱いしません。結果には、個別のファイル名や値を含めません。

In [ ]:
project = REPO / "tools/RealSaveImport/RealSaveImport.csproj"
command(
    ["dotnet", "build", project, "-c", "Release", "-p:UseNuGetRefs=true", "--nologo"],
    cwd=REPO, timeout=1200)
reader_output = command(
    ["dotnet", "run", "-c", "Release", "--project", project,
     "--no-build", "--no-launch-profile", "--", STAGE],
    cwd=REPO, private=True, timeout=1200)
lines = reader_output.splitlines()
if len(lines) < 2 or not lines[0].startswith("schema,originals,"):
    raise RuntimeError("RealSaveImportの出力形式が想定と異なります。")
reader = next(csv.DictReader(lines[:2]))
reader = {key: float(value) if "." in value else int(value)
          for key, value in reader.items()}
if (reader["originals"] != len(inputs)
    or reader["parsed_ok"] + reader["unidentifiable"] + reader["errors"] != len(inputs)
    or reader["errors"] != 0 or reader["unchanged"] != len(inputs)):
    raise AssertionError("読み取り結果・件数・元セーブ不変性の検証に失敗しました。")
display(pd.DataFrame([reader]).T.rename(columns={0: "値"}))
print("検証結果：読み取りエラー0・全ファイル不変。unidentifiableは別扱いです。")

## 5. コールド／ウォーム取り込みを測定

セーブ群を新しいプロファイルへ初回取り込みし、同じセーブの再走査がスキップされるか確認します。実行時間はColab VM固有の計測で、ゲーム内TPSではありません。

In [ ]:
benchmark_project = REPO / "tools/SaveImportBenchmark/SaveImportBenchmark.csproj"
command(
    ["dotnet", "build", benchmark_project, "-c", "Release", "-p:UseNuGetRefs=true", "--nologo"],
    cwd=REPO, timeout=1200)
benchmark_output = command(
    ["dotnet", "run", "-c", "Release", "--project", benchmark_project,
     "--no-build", "--no-launch-profile", "--", "--dir", STAGE, REPEATS],
    cwd=REPO, private=True, timeout=1800)
measurements = pd.read_csv(io.StringIO(benchmark_output))
required = {"files", "cold_read", "cold_failed", "unidentifiable", "warm_skipped",
            "warm_failed", "cold_ms", "warm_ms", "cold_alloc_bytes", "warm_alloc_bytes"}
if not required.issubset(measurements):
    raise RuntimeError("SaveImportBenchmarkの出力列が想定と異なります。")
if ((measurements["files"] != len(inputs)).any()
    or (measurements["cold_failed"] != 0).any()
    or (measurements["warm_failed"] != 0).any()
    or (measurements["unidentifiable"] != reader["unidentifiable"]).any()
    or (measurements["warm_skipped"] != len(inputs)).any()):
    raise AssertionError("取込に失敗、またはウォーム再走査がスキップされませんでした。")
if any(sha256(path) != digest for path, digest in source_hashes):
    raise AssertionError("コピー元のハッシュが変化しました。")
if any(sha256(STAGE / path.name) != digest for path, digest in source_hashes):
    raise AssertionError("読み取り用コピーが変化しました。")
display(measurements[["run", "files", "cold_ms", "warm_ms",
                       "cold_alloc_bytes", "warm_alloc_bytes"]].round(2))
measurements.to_csv(OUT / "save-import-measurements.csv", index=False)
print("初回取り込み中央値 (ms):", round(measurements["cold_ms"].median(), 2))
print("再走査中央値 (ms):", round(measurements["warm_ms"].median(), 2))

## 6. グラフと匿名レポート

生成するファイルには、個々のセーブ名・セーブ本文・ファイルハッシュを記録しません。ファイルはColabの左側「ファイル」から取得できます。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(measurements["run"], measurements["cold_ms"], marker="o", label="Cold")
ax.plot(measurements["run"], measurements["warm_ms"], marker="s", linestyle="--", label="Warm")
ax.set(xlabel="Run", ylabel="Time (ms)", title="Real save imports: cold vs warm")
ax.legend()
ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(OUT / "save-import-latency.png", dpi=150)
plt.show()
plt.close(fig)

report = {
    "schema": 1, "commit": COMMIT_SHA, "input_count": len(inputs),
    "input_bytes": bytes_total, "reader": reader,
    "benchmark": {
        "repeats": REPEATS,
        "cold_median_ms": float(measurements["cold_ms"].median()),
        "warm_median_ms": float(measurements["warm_ms"].median()),
        "cold_alloc_median_bytes": float(measurements["cold_alloc_bytes"].median()),
    }, "scope": "Offline SaveReader/ImportEngine only; not in-game TPS or UI"
}
(OUT / "save-analysis.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("レポート:", [p.name for p in sorted(OUT.glob("save-*"))])
print("注意: 入力コピーはセッション中 /content/rws-colab-private/saves に残ります。"
      "終了後はColabのランタイムを削除してください。")